# Publications HAL par laboratoire (2015–2025)

Interrogation de l'API HAL (`https://api.archives-ouvertes.fr/search/`) par `structId_i` pour chaque laboratoire.

In [ ]:
import requests
import pandas as pd
import time
from tqdm.notebook import tqdm

## Liste des laboratoires

In [ ]:
LABS = [
    ("CMP",           {244423}),
    ("CRAN",          {185180, 1001}),
    ("CREATIS",       {139739}),
    ("CRIL",          {90448, 1628}),
    ("CRISTAL",       {410272, 389110, 388977, 390300, 183073, 111636, 24885, 2546, 186929}),
    ("DI ENS",        {25027}),
    ("CROSSING (IRL)",{1063106}),
    ("ETIS",          {1003474, 1061575, 1087906, 1003348}),
    ("FILOFOCS (UMI)",{1006288, 1006289}),
    ("GIPSA-Lab",     {1043333, 1042376, 24470}),
    ("GREYC",         {150}),
    ("G-SCOP",        {1043137, 1041927, 74240}),
    ("HEUDIASYC",     {389870}),
    ("I3S",           {13009, 552896, 1079434}),
    ("ICUBE",         {217648, 1073080}),
    ("IDRIS",         {1823}),
    ("IPAL (IRL)",    {542003, 220880, 138926}),
    ("IRIF",          {1005016, 444497}),
    ("IRISA",         {491183, 491231, 490899, 491188, 1092618, 491177, 1092619, 490900,
                       419364, 419370, 105128, 2494, 25255, 419365, 419367, 491230,
                       419363, 419366, 491232, 419362, 1099404, 545024, 1099406,
                       1099401, 1099402, 1099435, 525233, 1088566, 1088569, 495900,
                       489780, 1092631, 1092630, 1092628, 1092632, 1092626, 1092625, 1092629}),
    ("IRIT",          {34499, 1082335}),
    ("ISIR",          {541937, 96164}),
    ("JFLI",          {542009, 229050}),
    ("L2S",           {1051117, 1289}),
    ("LAAS",          {459}),
    ("LABRI",         {3102}),
    ("LAB-STICC",     {486345, 491660, 199324, 81533, 1089048}),
    ("LAMIH",         {1067790, 1303}),
    ("LAMSADE",       {989}),
    ("LIG",           {1043301, 1041964, 24471}),
    ("LIGM",          {1001627, 3210}),
    ("LIMOS",         {1063677, 490706, 857}),
    ("LIP",           {35418}),
    ("LIP6",          {541703, 233, 1095103}),
    ("LIPN",          {1000994, 994, 1086916, 1056718}),
    ("LIRIS",         {2003, 1086665}),
    ("LIRMM",         {181, 1071941}),
    ("LIS",           {527033, 199402, 199394, 862, 178374}),
    ("LISN",          {1061259, 1041968, 247329, 2544, 1050003, 81750}),
    ("LIX",           {2071, 1041697, 1071530, 1070048}),
    ("LMF",           {1065710}),
    ("LORIA",         {206040, 466633}),
    ("LS2N",          {1088564, 473973, 95421, 1693, 21439}),
    ("LMF-LSV",       {1065710, 1042689, 2571}),
    ("MDLS",          {210816}),
    ("RELAX",         {1040410, 528907}),
    ("ROOT (IRL)",    {389700}),
    ("STMS",          {541779, 1374}),
    ("TIMA",          {1043043, 1044023, 640}),
    ("TIMC IMAG",     {1043049, 1070489, 1042061, 707, 574002, 555959, 1056575}),
    ("VERIMAG",       {1043148, 1041816, 194}),
]

## Fonction d'interrogation de l'API HAL

In [ ]:
HAL_API_URL = "https://api.archives-ouvertes.fr/search/"
YEAR_MIN    = 2015
YEAR_MAX    = 2025
BATCH_SIZE  = 1000
SLEEP_SEC   = 0.3


def fetch_lab_publications(lab_name: str, struct_ids: set,
                           year_min: int = YEAR_MIN,
                           year_max: int = YEAR_MAX) -> list[dict]:
    """
    Récupère toutes les publications HAL d'un laboratoire.
    Champs : halId, titre, auteurs, ORCID, DOI, année, type doc, langue,
             éditeur, revue, conférence.
    """
    struct_query = " OR ".join(str(sid) for sid in struct_ids)
    q  = f"structId_i:({struct_query})"
    fq = f"producedDateY_i:[{year_min} TO {year_max}]"

    records = []
    start   = 0

    while True:
        params = {
            "q":     q,
            "fq":    fq,
            "fl":    ("halId_s,title_s,authFullName_s,authOrcid_s,doiId_s,"
                      "producedDateY_i,docType_s,language_s,"
                      "publisher_s,journalTitle_s,conferenceTitle_s"),
            "rows":  BATCH_SIZE,
            "start": start,
            "wt":    "json",
        }

        resp = requests.get(HAL_API_URL, params=params, timeout=60)
        resp.raise_for_status()
        data = resp.json()

        docs      = data["response"]["docs"]
        num_found = data["response"]["numFound"]

        for doc in docs:
            title_field = doc.get("title_s")
            lang_field  = doc.get("language_s")

            names  = doc.get("authFullName_s") or []
            orcids = list(doc.get("authOrcid_s") or [])
            # authOrcid_s peut être plus court que authFullName_s
            orcids += [""] * max(0, len(names) - len(orcids))

            records.append({
                "laboratoire":   lab_name,
                "auteurs":       "; ".join(names),
                "orcids":        "; ".join(orcids[:len(names)]),
                "titre":         title_field[0] if title_field else None,
                "doi":           doc.get("doiId_s"),
                "annee":         doc.get("producedDateY_i"),
                "type_doc":      doc.get("docType_s"),
                "langue":        lang_field[0] if lang_field else None,
                "editeur":       doc.get("publisher_s"),
                "journal_revue": doc.get("journalTitle_s"),
                "conference":    doc.get("conferenceTitle_s"),
                "hal_id":        doc.get("halId_s"),
            })

        start += len(docs)
        if start >= num_found or not docs:
            break

        time.sleep(SLEEP_SEC)

    return records

## Collecte des données pour tous les laboratoires

In [ ]:
all_records = []
errors      = []

for lab_name, struct_ids in tqdm(LABS, desc="Laboratoires"):
    try:
        records = fetch_lab_publications(lab_name, struct_ids)
        all_records.extend(records)
        print(f"  {lab_name:20s} → {len(records):>5} publications")
    except Exception as e:
        errors.append((lab_name, str(e)))
        print(f"  ⚠ Erreur pour {lab_name}: {e}")

print(f"\nTotal brut : {len(all_records)} enregistrements")
if errors:
    print(f"Laboratoires en erreur : {[e[0] for e in errors]}")

## Construction du dataframe global

In [ ]:
DF_COLS = ["laboratoire", "auteurs", "orcids", "titre", "doi", "annee",
           "type_doc", "langue", "editeur", "journal_revue", "conference", "hal_id"]

df = pd.DataFrame(all_records, columns=DF_COLS)
df["annee"] = pd.to_numeric(df["annee"], errors="coerce").astype("Int64")

print(f"Shape      : {df.shape}")
print(f"Années     : {sorted(df['annee'].dropna().unique().tolist())}")
print(f"Labos      : {df['laboratoire'].nunique()}")
print(f"Types doc  : {sorted(df['type_doc'].dropna().unique().tolist())}")
print(f"Langues    : {sorted(df['langue'].dropna().unique().tolist())}")
print(f"Avec ORCID : {(df['orcids'].str.replace('; ','').str.strip() != '').sum()} publications")
df.head(10)

## Dédoublonnage + restauration des résultats OpenAlex

Les résultats OpenAlex sont **restaurés depuis le fichier `hal_openalex_uniques.csv`** (déjà calculé) sans relancer les requêtes API.

In [ ]:
# ── Publications uniques ─────────────────────────────────────────────────────
df_unique = df.drop_duplicates(subset=["hal_id"]).reset_index(drop=True)
print(f"Publications uniques : {len(df_unique)}")

# ── Restauration des résultats OpenAlex depuis le CSV existant ───────────────
OA_CACHE  = "hal_openalex_uniques.csv"
OA_RESULT_COLS = ["hal_id", "in_openalex", "match_method",
                  "openalex_id", "openalex_title", "title_sim"]

try:
    oa_cache = pd.read_csv(
        OA_CACHE,
        usecols=lambda c: c in OA_RESULT_COLS,
        encoding="utf-8-sig",
        dtype={"in_openalex": "boolean"},
    )
    df_unique = df_unique.merge(oa_cache, on="hal_id", how="left")
    df_unique["in_openalex"] = df_unique["in_openalex"].fillna(False)
    n = df_unique["in_openalex"].sum()
    print(f"Résultats OpenAlex restaurés depuis '{OA_CACHE}' : {n} publications trouvées")
except FileNotFoundError:
    print(f"⚠  '{OA_CACHE}' introuvable — exécuter la section OpenAlex pour le générer.")
    df_unique["in_openalex"]    = False
    df_unique["match_method"]   = pd.NA
    df_unique["openalex_id"]    = pd.NA
    df_unique["openalex_title"] = pd.NA
    df_unique["title_sim"]      = pd.NA

# ── Propagation vers df (par labo) ───────────────────────────────────────────
OA_JOIN_COLS = ["hal_id", "in_openalex", "match_method", "openalex_id"]
df = df.drop(columns=[c for c in OA_JOIN_COLS[1:] if c in df.columns], errors="ignore")
df = df.merge(df_unique[OA_JOIN_COLS], on="hal_id", how="left")

print(f"df shape après enrichissement : {df.shape}")

---
# Croisement avec OpenAlex

Stratégie en deux passes sur `df_unique` :
1. **Passe DOI** : requête par lot (50 DOI à la fois) via `filter=doi:…`
2. **Passe titre** : pour les publications sans DOI (ou non trouvées par DOI), recherche sur `display_name.search` avec vérification de similarité (seuil 0.85)

> **Note** : la recherche par titre est heuristique — des faux positifs sont possibles sur des titres courts ou très génériques.
>
> Si les résultats OpenAlex ont déjà été calculés et exportés, **ne pas ré-exécuter les cellules de cette section** — les résultats sont restaurés automatiquement depuis le CSV.

In [ ]:
import difflib
import unicodedata
import re

# ── Paramètres OpenAlex ──────────────────────────────────────────────────────
OPENALEX_URL    = "https://api.openalex.org/works"
OPENALEX_MAILTO = "votre.email@example.com"   # remplacer pour le « polite pool »
OA_BATCH_SIZE   = 50     # max DOI par requête (recommandé par OpenAlex)
OA_SLEEP_SEC    = 0.1    # pause entre requêtes
TITLE_SIM_MIN   = 0.85   # seuil de similarité pour la correspondance par titre


def _norm_title(title: str) -> str:
    """Normalise un titre pour la comparaison : minuscules, sans accents ni ponctuations."""
    if not title:
        return ""
    t = unicodedata.normalize("NFD", title.lower())
    t = "".join(c for c in t if unicodedata.category(c) != "Mn")
    t = re.sub(r"[^a-z0-9\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()


def _norm_doi(doi: str) -> str:
    """Renvoie le DOI sans le préfixe URI, en minuscules."""
    if not doi:
        return ""
    doi = doi.strip().lower()
    for prefix in ("https://doi.org/", "http://doi.org/", "doi:"):
        if doi.startswith(prefix):
            doi = doi[len(prefix):]
    return doi


def _oa_params(**extra) -> dict:
    return {"mailto": OPENALEX_MAILTO, "select": "id,doi,display_name", **extra}


def lookup_dois(doi_list: list[str]) -> dict[str, dict]:
    """Recherche par lots de DOIs. Renvoie {doi_normalisé -> {...}}."""
    results = {}
    for i in range(0, len(doi_list), OA_BATCH_SIZE):
        batch = doi_list[i : i + OA_BATCH_SIZE]
        filter_val = "|".join(batch)
        params = _oa_params(filter=f"doi:{filter_val}", per_page=OA_BATCH_SIZE)
        try:
            resp = requests.get(OPENALEX_URL, params=params, timeout=30)
            resp.raise_for_status()
            for work in resp.json().get("results", []):
                raw_doi = work.get("doi") or ""
                nd = _norm_doi(raw_doi)
                if nd:
                    results[nd] = {
                        "openalex_id":    work.get("id"),
                        "openalex_title": work.get("display_name"),
                    }
        except Exception as e:
            print(f"  ⚠ Erreur batch DOI [{i}:{i+OA_BATCH_SIZE}] : {e}")
        time.sleep(OA_SLEEP_SEC)
    return results


def lookup_title(title: str) -> dict | None:
    """Recherche par titre avec vérification de similarité."""
    if not title or len(title.strip()) < 10:
        return None
    search_title = title[:200]
    params = _oa_params(**{"filter": f'display_name.search:"{search_title}"', "per_page": 1})
    try:
        resp = requests.get(OPENALEX_URL, params=params, timeout=30)
        resp.raise_for_status()
        results = resp.json().get("results", [])
        if not results:
            return None
        work = results[0]
        oa_title = work.get("display_name") or ""
        sim = difflib.SequenceMatcher(
            None, _norm_title(title), _norm_title(oa_title)
        ).ratio()
        if sim >= TITLE_SIM_MIN:
            return {
                "openalex_id":    work.get("id"),
                "openalex_title": oa_title,
                "title_sim":      round(sim, 3),
            }
    except Exception as e:
        print(f"  ⚠ Erreur titre : {e}")
    return None

In [ ]:
# ── Initialisation des colonnes résultats ────────────────────────────────────
df_unique = df_unique.copy()
df_unique["in_openalex"]    = False
df_unique["match_method"]   = pd.NA
df_unique["openalex_id"]    = pd.NA
df_unique["openalex_title"] = pd.NA
df_unique["title_sim"]      = pd.NA

# ── Passe 1 : DOI ────────────────────────────────────────────────────────────
has_doi = df_unique["doi"].notna() & (df_unique["doi"].str.strip() != "")
dois_norm = df_unique.loc[has_doi, "doi"].apply(_norm_doi).tolist()
doi_index = df_unique.loc[has_doi].index

print(f"Publications avec DOI : {has_doi.sum()} / {len(df_unique)}")
print("Recherche par DOI dans OpenAlex…")

doi_results = lookup_dois(dois_norm)
print(f"  → {len(doi_results)} DOI trouvés dans OpenAlex")

for idx, doi_raw in zip(doi_index, dois_norm):
    if doi_raw in doi_results:
        hit = doi_results[doi_raw]
        df_unique.at[idx, "in_openalex"]    = True
        df_unique.at[idx, "match_method"]   = "doi"
        df_unique.at[idx, "openalex_id"]    = hit["openalex_id"]
        df_unique.at[idx, "openalex_title"] = hit["openalex_title"]

print(f"Publications trouvées via DOI : {(df_unique['match_method'] == 'doi').sum()}")

In [ ]:
# ── Passe 2 : titre ──────────────────────────────────────────────────────────
to_search_by_title = df_unique[
    (~df_unique["in_openalex"]) & (df_unique["titre"].notna())
].index

print(f"Publications à rechercher par titre : {len(to_search_by_title)}")
print("Recherche par titre dans OpenAlex (peut être long)…")

for idx in tqdm(to_search_by_title, desc="Recherche titre"):
    titre = df_unique.at[idx, "titre"]
    hit = lookup_title(titre)
    if hit:
        df_unique.at[idx, "in_openalex"]    = True
        df_unique.at[idx, "match_method"]   = "titre"
        df_unique.at[idx, "openalex_id"]    = hit["openalex_id"]
        df_unique.at[idx, "openalex_title"] = hit["openalex_title"]
        df_unique.at[idx, "title_sim"]      = hit["title_sim"]
    time.sleep(OA_SLEEP_SEC)

print(f"Publications trouvées via titre : {(df_unique['match_method'] == 'titre').sum()}")
print(f"Publications non trouvées : {(~df_unique['in_openalex']).sum()}")

## Taux de couverture global (df_unique)

In [ ]:
total   = len(df_unique)
found   = df_unique["in_openalex"].sum()
by_doi  = (df_unique["match_method"] == "doi").sum()
by_titl = (df_unique["match_method"] == "titre").sum()

print("=" * 50)
print(f"Publications HAL uniques           : {total:>7}")
print(f"Présentes dans OpenAlex            : {found:>7}  ({100*found/total:.1f}%)")
print(f"  dont trouvées par DOI            : {by_doi:>7}  ({100*by_doi/total:.1f}%)")
print(f"  dont trouvées par titre          : {by_titl:>7}  ({100*by_titl/total:.1f}%)")
print(f"Non trouvées dans OpenAlex         : {total-found:>7}  ({100*(total-found)/total:.1f}%)")
print("=" * 50)

df_unique.groupby("match_method", dropna=False).size().rename("count").to_frame()

## Taux de couverture par laboratoire

In [ ]:
# On repart de df (labo × publication) et on joint df_unique (résultats OA)
# pour être sûr d'avoir les bonnes valeurs de match_method et in_openalex.
df_labo = (
    df[["laboratoire", "hal_id", "doi"]]
    .merge(
        df_unique[["hal_id", "in_openalex", "match_method"]],
        on="hal_id", how="left"
    )
    .assign(_has_doi=lambda d: d["doi"].notna() & (d["doi"].str.strip() != ""))
)

stats_labo = (
    df_labo
    .groupby("laboratoire")
    .agg(
        n_hal      =("hal_id",       "count"),
        n_openalex =("in_openalex",  "sum"),
        n_doi      =("match_method", lambda x: (x == "doi").sum()),
        n_titre    =("match_method", lambda x: (x == "titre").sum()),
        n_avec_doi =("_has_doi",     "sum"),
        n_sans_doi =("_has_doi",     lambda x: (~x).sum()),
    )
    .assign(
        taux_global=lambda d: (d["n_openalex"] / d["n_hal"]      * 100).round(1),
        taux_doi   =lambda d: (d["n_doi"]      / d["n_avec_doi"] * 100).round(1),
        taux_titre =lambda d: (d["n_titre"]    / d["n_sans_doi"] * 100).round(1),
    )
    .sort_values("taux_global", ascending=False)
)

# Vérification : le total n_titre doit être >= (df_unique['match_method']=='titre').sum()
print(f"Total n_titre dans stats_labo : {stats_labo['n_titre'].sum()}")
print(f"Référence df_unique           : {(df_unique['match_method'] == 'titre').sum()}")
stats_labo

---
# Analyses de couverture par dimension

Les analyses suivantes utilisent `df_unique` (publications dédoublonnées) pour éviter de compter plusieurs fois une même publication affectée à plusieurs laboratoires.

## Couverture par année de publication

In [ ]:
def coverage_table(df_: pd.DataFrame, col: str) -> pd.DataFrame:
    """Calcule le taux de couverture OpenAlex pour chaque valeur d'une colonne.
    - taux_global : % de publications HAL trouvées dans OpenAlex (/ n_hal)
    - taux_doi    : parmi les pubs HAL avec DOI, % retrouvées dans OpenAlex (/ n_avec_doi)
    - taux_titre  : parmi les pubs HAL sans DOI, % retrouvées dans OpenAlex (/ n_sans_doi)
    """
    has_doi = df_["doi"].notna() & (df_["doi"].str.strip() != "")
    return (
        df_.assign(_has_doi=has_doi)
        .groupby(col, dropna=False)
        .agg(
            n_hal      =("hal_id",       "count"),
            n_openalex =("in_openalex",  "sum"),
            n_doi      =("match_method", lambda x: (x == "doi").sum()),
            n_titre    =("match_method", lambda x: (x == "titre").sum()),
            n_avec_doi =("_has_doi",     "sum"),
            n_sans_doi =("_has_doi",     lambda x: (~x).sum()),
        )
        .assign(
            taux_global=lambda d: (d["n_openalex"] / d["n_hal"]      * 100).round(1),
            taux_doi   =lambda d: (d["n_doi"]      / d["n_avec_doi"] * 100).round(1),
            taux_titre =lambda d: (d["n_titre"]    / d["n_sans_doi"] * 100).round(1),
        )
        .sort_index()
    )


stats_annee = coverage_table(df_unique, "annee")
print("Couverture OpenAlex par année de publication")
stats_annee

## Couverture par type de document (docType_s)

In [ ]:
stats_type = (
    coverage_table(df_unique, "type_doc")
    .sort_values("taux_global", ascending=False)
)
print("Couverture OpenAlex par type de document")
stats_type

## Couverture par langue de publication (language_s)

In [ ]:
stats_langue = (
    coverage_table(df_unique, "langue")
    .sort_values("taux_global", ascending=False)
)
print("Couverture OpenAlex par langue de publication")
stats_langue

---
## Export final

In [ ]:
df_unique.to_csv("hal_openalex_uniques.csv",          index=False, encoding="utf-8-sig")
df.to_csv(       "hal_openalex_par_labo.csv",         index=False, encoding="utf-8-sig")
stats_labo.to_csv(  "couverture_par_labo.csv",                     encoding="utf-8-sig")
stats_annee.to_csv( "couverture_par_annee.csv",                    encoding="utf-8-sig")
stats_type.to_csv(  "couverture_par_type_doc.csv",                 encoding="utf-8-sig")
stats_langue.to_csv("couverture_par_langue.csv",                   encoding="utf-8-sig")

print("Fichiers exportés :")
print("  hal_openalex_uniques.csv      (df_unique enrichi)")
print("  hal_openalex_par_labo.csv     (df non dédoublonné enrichi)")
print("  couverture_par_labo.csv")
print("  couverture_par_annee.csv")
print("  couverture_par_type_doc.csv")
print("  couverture_par_langue.csv")

---
# Analyse des auteurs d'un laboratoire pour une année donnée

Comparaison des auteurs obtenus via deux sources :
- **Solution A** : requête directe à OpenAlex sur l'institution
- **Solution B** : données HAL déjà collectées (`df`)

La cellule de comparaison identifie les auteurs présents dans HAL mais absents d'OpenAlex, et inversement.

In [ ]:
# ── Paramètres ───────────────────────────────────────────────────────────────
LABO_CIBLE  = "LIS"
ANNEE_CIBLE = 2022

# ════════════════════════════════════════════════════════════════════════════
# SOLUTION A : auteurs via OpenAlex (requête directe sur l'institution)
# ════════════════════════════════════════════════════════════════════════════

# Étape A1 – chercher le laboratoire par son acronyme dans OpenAlex
resp = requests.get(
    "https://api.openalex.org/institutions",
    params={
        "search":   LABO_CIBLE,
        "filter":   "country_code:FR",
        "per_page": 10,
        "mailto":   OPENALEX_MAILTO,
    },
    timeout=30,
)
resp.raise_for_status()
institutions = resp.json().get("results", [])

print("Institutions trouvées — choisir l'ID à utiliser :")
for inst in institutions:
    print(f"  {inst['id']}  |  {inst['display_name']}  |  {inst.get('ror', '?')}")

# ── Mettre ici l'ID OpenAlex correspondant au laboratoire ────────────────────
# (ajuster si nécessaire après lecture des résultats ci-dessus)
OA_LIS_ID = institutions[0]["id"] if institutions else None
print(f"\nID retenu : {OA_LIS_ID}")

In [ ]:
# Étape A2 – récupérer tous les travaux de LIS en ANNEE_CIBLE et extraire les auteurs
# (cursor-based pagination pour éviter la limite de 10 000 résultats)
authors_A = {}   # {nom_normalisé: nom_original}
cursor = "*"
n_works = 0

while cursor:
    params = {
        "filter":   f"institutions.id:{OA_LIS_ID},publication_year:{ANNEE_CIBLE}",
        "select":   "id,authorships",
        "per_page": 200,
        "cursor":   cursor,
        "mailto":   OPENALEX_MAILTO,
    }
    resp = requests.get(OPENALEX_URL, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    for work in data.get("results", []):
        n_works += 1
        for authorship in work.get("authorships", []):
            name = (authorship.get("author") or {}).get("display_name", "").strip()
            if name:
                authors_A[_norm_title(name)] = name   # _norm_title sert de clé normalisée

    cursor = data.get("meta", {}).get("next_cursor")
    time.sleep(OA_SLEEP_SEC)

authors_A_list = sorted(authors_A.values(), key=lambda n: n.split()[-1])  # tri par nom

print(f"Publications LIS {ANNEE_CIBLE} trouvées dans OpenAlex : {n_works}")
print(f"Auteurs uniques (Solution A) : {len(authors_A_list)}")
print()
for a in authors_A_list:
    print(f"  {a}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# SOLUTION B : auteurs via les données HAL (df)
# ════════════════════════════════════════════════════════════════════════════

mask = (df["laboratoire"] == LABO_CIBLE) & (df["annee"] == ANNEE_CIBLE)
n_pubs_B = mask.sum()

authors_B = {}   # {nom_normalisé: nom_original}
for row in df.loc[mask, "auteurs"].dropna():
    for name in row.split("; "):
        name = name.strip()
        if name:
            authors_B[_norm_title(name)] = name

authors_B_list = sorted(authors_B.values(), key=lambda n: n.split()[-1])

print(f"Publications LIS {ANNEE_CIBLE} dans HAL : {n_pubs_B}")
print(f"Auteurs uniques (Solution B) : {len(authors_B_list)}")
print()
for a in authors_B_list:
    print(f"  {a}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# COMPARAISON : dataframe de tous les auteurs avec présence HAL / OpenAlex
# ════════════════════════════════════════════════════════════════════════════

# Union de toutes les clés normalisées
all_keys = set(authors_A) | set(authors_B)

rows = []
for key in all_keys:
    rows.append({
        "auteur":      authors_B.get(key) or authors_A.get(key),  # nom préféré depuis HAL
        "dans_HAL":    key in authors_B,
        "dans_OpenAlex": key in authors_A,
    })

df_auteurs = (
    pd.DataFrame(rows)
    .sort_values(["dans_HAL", "dans_OpenAlex", "auteur"],
                 ascending=[False, False, True])
    .reset_index(drop=True)
)

print(f"Auteurs LIS {ANNEE_CIBLE} — récapitulatif")
print(f"  Dans HAL et OpenAlex  : {(df_auteurs['dans_HAL'] & df_auteurs['dans_OpenAlex']).sum()}")
print(f"  Dans HAL seulement    : {(df_auteurs['dans_HAL'] & ~df_auteurs['dans_OpenAlex']).sum()}")
print(f"  Dans OpenAlex seulement: {(~df_auteurs['dans_HAL'] & df_auteurs['dans_OpenAlex']).sum()}")
print(f"  Total                 : {len(df_auteurs)}")
df_auteurs

In [ ]:
output_auteurs = f"auteurs_{LABO_CIBLE}_{ANNEE_CIBLE}.csv"
df_auteurs.to_csv(output_auteurs, index=False, encoding="utf-8-sig")
print(f"Fichier exporté : {output_auteurs}")

---
## Analyse des auteurs par ORCID

Comparaison HAL ↔ OpenAlex basée sur les identifiants ORCID plutôt que sur les noms, ce qui évite les problèmes de découpage prénom / nom.

- Les auteurs **avec ORCID** sont vérifiés directement via l'API OpenAlex `/authors`.
- Les auteurs **sans ORCID** dans HAL restent comparés par nom normalisé (via la liste `authors_A` de la Solution A).

In [ ]:
# ── Extraction des ORCIDs HAL pour LABO_CIBLE / ANNEE_CIBLE ──────────────────
mask = (df["laboratoire"] == LABO_CIBLE) & (df["annee"] == ANNEE_CIBLE)

orcid_to_name = {}    # {orcid_brut -> nom HAL}
name_no_orcid = {}    # {nom_normalisé -> nom HAL} pour auteurs sans ORCID

for _, row in df[mask].iterrows():
    names  = [n.strip() for n in (row["auteurs"] or "").split("; ") if n.strip()]
    orcids = [o.strip() for o in (row["orcids"]  or "").split("; ")]
    orcids += [""] * max(0, len(names) - len(orcids))   # padding si nécessaire

    for name, orcid in zip(names, orcids):
        if orcid:
            orcid_to_name[orcid] = name
        else:
            name_no_orcid[_norm_title(name)] = name

print(f"Auteurs HAL {LABO_CIBLE} {ANNEE_CIBLE}")
print(f"  Avec ORCID : {len(orcid_to_name)}")
print(f"  Sans ORCID : {len(name_no_orcid)}")
print()
print("Auteurs avec ORCID :")
for orcid, name in sorted(orcid_to_name.items(), key=lambda x: x[1].split()[-1]):
    print(f"  {name:40s}  {orcid}")

In [ ]:
# ── Recherche des ORCIDs dans OpenAlex (par lots) ────────────────────────────
def norm_orcid(orcid: str) -> str:
    """Normalise l'ORCID au format URI attendu par OpenAlex."""
    orcid = orcid.strip()
    if not orcid.startswith("https://"):
        orcid = f"https://orcid.org/{orcid}"
    return orcid


orcid_list   = list(orcid_to_name.keys())
found_orcids = set()   # ORCIDs (bruts) trouvés dans OpenAlex

for i in range(0, len(orcid_list), OA_BATCH_SIZE):
    batch      = [norm_orcid(o) for o in orcid_list[i : i + OA_BATCH_SIZE]]
    filter_val = "|".join(batch)
    params = {
        "filter":   f"orcid:{filter_val}",
        "select":   "id,orcid,display_name",
        "per_page": OA_BATCH_SIZE,
        "mailto":   OPENALEX_MAILTO,
    }
    try:
        resp = requests.get("https://api.openalex.org/authors",
                            params=params, timeout=30)
        resp.raise_for_status()
        for author in resp.json().get("results", []):
            raw = (author.get("orcid") or "").replace("https://orcid.org/", "")
            if raw:
                found_orcids.add(raw)
    except Exception as e:
        print(f"  ⚠ Erreur lot {i} : {e}")
    time.sleep(OA_SLEEP_SEC)

print(f"ORCIDs trouvés dans OpenAlex : {len(found_orcids)} / {len(orcid_list)}")
print(f"ORCIDs HAL non trouvés       : {len(orcid_list) - len(found_orcids)}")

In [ ]:
# ── Dataframe de comparaison ORCID ───────────────────────────────────────────
rows = []

# Auteurs AVEC ORCID dans HAL
for orcid, name in orcid_to_name.items():
    rows.append({
        "auteur":        name,
        "orcid":         orcid,
        "dans_HAL":      True,
        "dans_OpenAlex": orcid in found_orcids,
        "methode":       "orcid",
    })

# Auteurs SANS ORCID dans HAL : fallback comparaison par nom (via authors_A)
for key, name in name_no_orcid.items():
    rows.append({
        "auteur":        name,
        "orcid":         "",
        "dans_HAL":      True,
        "dans_OpenAlex": key in authors_A,
        "methode":       "nom",
    })

df_auteurs_orcid = (
    pd.DataFrame(rows)
    .sort_values(["methode", "dans_OpenAlex", "auteur"],
                 ascending=[True, False, True])
    .reset_index(drop=True)
)

# Récapitulatif
mask_orcid = df_auteurs_orcid["methode"] == "orcid"
mask_nom   = df_auteurs_orcid["methode"] == "nom"
print(f"Auteurs {LABO_CIBLE} {ANNEE_CIBLE} — comparaison par ORCID")
print(f"  Avec ORCID → trouvés dans OpenAlex     : "
      f"{(mask_orcid &  df_auteurs_orcid['dans_OpenAlex']).sum()}")
print(f"  Avec ORCID → absents d'OpenAlex        : "
      f"{(mask_orcid & ~df_auteurs_orcid['dans_OpenAlex']).sum()}")
print(f"  Sans ORCID → trouvés par nom           : "
      f"{(mask_nom   &  df_auteurs_orcid['dans_OpenAlex']).sum()}")
print(f"  Sans ORCID → absents d'OpenAlex        : "
      f"{(mask_nom   & ~df_auteurs_orcid['dans_OpenAlex']).sum()}")
print(f"  Total                                  : {len(df_auteurs_orcid)}")

df_auteurs_orcid

In [ ]:
output_orcid = f"auteurs_orcid_{LABO_CIBLE}_{ANNEE_CIBLE}.csv"
df_auteurs_orcid.to_csv(output_orcid, index=False, encoding="utf-8-sig")
print(f"Fichier exporté : {output_orcid}")

---
## Origine des publications HAL absentes d'OpenAlex

Analyse des éditeurs, revues et conférences pour les publications non trouvées dans OpenAlex (sur l'ensemble de `df_unique`).

In [ ]:
not_in_oa = df_unique[~df_unique["in_openalex"]].copy()
TOP_N = 20

print(f"Publications HAL non trouvées dans OpenAlex : {len(not_in_oa)} "
      f"/ {len(df_unique)} ({100*len(not_in_oa)/len(df_unique):.1f}%)\n")

def top_table(series: pd.Series, label: str, n: int = TOP_N) -> pd.DataFrame:
    return (
        series.value_counts(dropna=False)
        .head(n)
        .rename("n_publications")
        .to_frame()
        .assign(pct=lambda d: (d["n_publications"] / len(not_in_oa) * 100).round(1))
    )

print(f"── Top {TOP_N} éditeurs ──────────────────────────────")
display(top_table(not_in_oa["editeur"], "éditeur"))

print(f"\n── Top {TOP_N} revues ───────────────────────────────")
display(top_table(not_in_oa["journal_revue"], "revue"))

print(f"\n── Top {TOP_N} conférences ──────────────────────────")
display(top_table(not_in_oa["conference"], "conférence"))